In [24]:
import pandas as pd

# Job-Task weights
job_task_input_mean_estimates = pd.read_csv(
    "https://huggingface.co/datasets/MIT-WAL/job_task_input_share/resolve/main/task_labor_input_mean_estimates/30_2/ONET_30_2_weight_mode_STANDARD_task_labor_input_mean_estimates.csv"
)

# Exposure scores
path = "./data/exposure_score/full_labelset.tsv"
exposure_df = pd.read_csv(path, sep='\t')
exposure_df = exposure_df[["Task ID", "beta"]]



In [25]:
crosswalk_df = pd.read_csv("./data/job_task_map/2019_to_SOC_Crosswalk.csv")
crosswalk_df = crosswalk_df.iloc[:, [0,2]].copy()
crosswalk_df.dropna(inplace=True)
crosswalk_df.columns = ["onetsoc_code", "soc_code"]

In [26]:
job_task_input_mean_estimates = job_task_input_mean_estimates.merge(crosswalk_df, on="onetsoc_code", how="left")

#Normalise pi by soc_code
job_task_input_mean_estimates["pi"] = job_task_input_mean_estimates.groupby("soc_code")["pi"].transform(lambda x: x / x.sum())

#Drop onetsoc_code as we will not need it anymore
job_task_input_mean_estimates.drop(columns=["onetsoc_code"], inplace=True)

In [30]:
print(job_task_input_mean_estimates.groupby("soc_code")["pi"].sum().describe())

count    7.740000e+02
mean     1.000000e+00
std      9.525255e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: pi, dtype: float64


In [ ]:
# Calculate job exposure scores
job_task_input_mean_estimates =pd.merge(job_task_input_mean_estimates, exposure_df, left_on="task_id", right_on="Task ID", how="left")
job_task_input_mean_estimates["weighted_beta"] = job_task_input_mean_estimates["beta"] * job_task_input_mean_estimates["pi"]
job_task_input_mean_estimates = job_task_input_mean_estimates.groupby("soc_code")["weighted_beta"].sum()

job_task_input_mean_estimates.to_csv("./data/exposure_score/job_exposure_scores.csv")